In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Glow Guide is a Multi-Agent Personal Care AI System with Google Agent Development Kit (ADK) and Gemini LLM

This notebook demonstrates a state-of-the-art modular agent system for personalized skincare, grooming, and wellness, following the orchestration pattern.

- Each agent is a callable tool.
- The root coordinator agent orchestrates workflow across agents.
- User input is interactive, and all agent reasoning uses Gemini 2.0 Flash.
- All competition requirements (multi-agent, parallel/sequential, LLM-powered, session/memory, observability) are addressable.


## Load Google API Key from Kaggle Secrets

This code cell loads the Google API Key securely from Kaggle secrets.  
**Requirement:** You must add your key in Kaggle's "Secrets" as `GOOGLE_API_KEY`.


In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")

🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 102213465 and label GOOGLE_API_KEY.'], 'error': {'code': 5}, 'wasSuccessful': False}.


## Import Required Libraries

This cell imports the Google Agent Development Kit (ADK) modules and Python utilities for agent orchestration, Gemini LLM, and logging.  
If needed, install with `!pip install google-agent-kit`.

In [3]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### OnboardingAgent (AgentTool)

- Role: Interactively collects all essential information from the user.
- What it does: Prompts for gender, city, skin type, diet, and—if applicable—menstrual cycle.  
- Result: Stores user profile as `user_profile` in session for downstream agents.

In [ ]:
onboarding_agent = Agent(
    name="OnboardingAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You are an onboarding specialist for a personal care platform.
    Ask the user, one by one:
    1. Gender (male/female/other)
    2. City
    3. Skin type (oily/dry/combination/normal)
    4. Diet (veg/non-veg)
    If gender is female, also ask:
    5. Menstrual cycle phase (follicular/luteal/menstrual)
    Store results as user_profile in the session.
    Return user_profile.
    """
)

### ProductAgent (AgentTool)

- Role: Recommends climate-adaptive, local skincare and personal care products using Gemini LLM.
- What it does: Reads the user profile from session, consults Gemini, and stores recommendations as `products`.
- Output: List of product names best-matched for user's context.

In [ ]:
product_agent = Agent(
    name="ProductAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You are a product recommendation expert.
    Using the user_profile from the session, recommend 2-3 local and climate-appropriate skincare/personal care products.
    Return results as a list of product names.
    """
)

### RoutineAgent (AgentTool)

- Role: Builds the personalized self-care routine.
- What it does: Uses both user profile and recommended products to design a morning and night routine.
- Output: Two lists, one for morning and one for night, stored as `routine`.

In [ ]:
routine_agent = Agent(
    name="RoutineAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You design complete morning and night routines.
    Use the user_profile and the recommended products from the session.
    Output two lists: morning_routine and night_routine.
    """
)

### NutritionAgent (AgentTool)

- Role: Provides personalized nutrition tips to support skin/hair health.
- What it does: Consults Gemini with the user profile for science-backed dietary advice, and stores tips as `nutrition_advice`.
- Output: List of specific nutrition recommendations.

In [ ]:
nutrition_agent = Agent(
    name="NutritionAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You provide nutrition advice for skin/hair/overall glow.
    Use user_profile from the session for personalized tips. Return as a short list or key tips.
    """
)

### HolisticHealthAgent (AgentTool)

- Role: Adapts routine for female users based on their reported menstrual cycle phase.
- What it does: If cycle info is present in profile, asks Gemini for relevant health/routine adaptations and stores as routine notes.
- Output: Text summary and routine note adjustments.

In [ ]:
holistic_health_agent = Agent(
    name="HolisticHealthAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    If the user_profile includes a cycle_phase, briefly explain key skin/hair care considerations during this phase and adapt the routine if needed.
    """
)

### HabitAgent (AgentTool)

- Role: Suggests actionable micro-beauty habits for daily use.
- What it does: Reviews the user's routine and personal context, then outputs three habit suggestions (stored as `habits`).
- Output: 3 easy-to-implement beauty/wellness habits.

In [ ]:
habit_agent = Agent(
    name="HabitAgent",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You suggest three daily micro-beauty habits based on the user's routine.
    """
)

### Root Coordinator Agent

- Role: Orchestrates the workflow by calling all sub-agents as tools.
- What it does: Sequences onboarding, product and nutrition agents (in parallel), then routine, holistic health, and habit agents.
- Output: Presents complete, personalized results to the user.
- Follows competition requirement for multi-agent orchestration, parallelism, LLM-powered reasoning, memory, and observability.

In [ ]:
root_agent = Agent(
    name="PersonalCareCoordinator",
    model=Gemini(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY),
    instruction="""
    You are a multi-agent personal care workflow orchestrator.
    Workflow:
    1. Call onboarding_agent to collect user profile.
    2. In parallel, call product_agent and nutrition_agent.
    3. Call routine_agent (uses user_profile and products).
    4. If applicable, call holistic_health_agent to adapt routines based on cycle_phase.
    5. Call habit_agent for micro-habit suggestions.
    6. Present all findings clearly to the user.
    Each sub-agent is available as a tool.
    """,
    tools=[
        AgentTool(onboarding_agent),
        AgentTool(product_agent),
        AgentTool(routine_agent),
        AgentTool(nutrition_agent),
        AgentTool(holistic_health_agent),
        AgentTool(habit_agent)
    ]
)

print("✅ root_agent created.")

## Execute Multi-Agent Workflow

This cell:
- Creates an interactive session.
- Runs the root coordinator, which sequentially/parallelly calls all agent tools according to the workflow, collecting user inputs where needed.
- All agent actions, memory updates, and traces are managed by ADK’s session and observability tools.

In [ ]:
session = Session(
    api_key=GOOGLE_API_KEY,
    memory_enabled=True,
    observability_enabled=True
)

# Start full workflow — user will interactively provide answers through onboarding agent.
result = root_agent(session)

## Final Results & Agent Execution Trace

Displays:
- Personalized routine
- Recommended products
- Nutrition advice
- Micro-habits
- Cycle-phase notes
- Full memory/trace log of all sub-agent actions

Use for analysis, evaluation, and competition reporting.

In [ ]:
print("\n==== FINAL PERSONALIZED OUTPUT ====\n")
personalized_response = result["response"] if isinstance(result, dict) else result
print(personalized_response)

print("\n== SESSION LOG ==")
for entry in session.memory.trace():
    print(entry)

# Discussion

This notebook demonstrates modern agent composition and orchestration using Google ADK and Gemini 2.0 Flash.  
- Each feature agent is a tool called by a root workflow agent.
- All reasoning is LLM-powered, user input is interactive.
- Parallel/sequential logic, session/memory, and observability are showcased.
- The system is modular for easy expansion (add more feature agents or tools as needed).
- Ready for further extension: agent evaluation, visualizations, integrations.

**Reference:** Kaggle ADK Orchestration [Agent architecture notebook](https://www.kaggle.com/code/kaggle5daysofai/day-1b-agent-architectures)